# Final Projesi - H(t) -> H(t+1) Enerji Tahmini, ACE/FACE ve Dijital Twin Simülasyonu

Bu notebook, mevcut `WholeBuilding_EnergyConsumptionDataSet.xlsx` verisini baz alarak final proje omurgasını kurar.

Ana hedefler:

- `H(t)` anındaki devre yükleri ve takvim bağlamından `H(t+1)` enerji tüketimini tahmin etmek.
- Tahmini spend-zone sınıfına çevirmek.
- Kullanıcı hedef zone seçerse ACE ve FACE mantığıyla uygulanabilir azaltım aksiyonları üretmek.
- Aksiyonlar uygulandıktan sonra aynı eğitilmiş modelle tekrar skorlayıp DT üzerinden önce/sonra sonucunu simüle etmek.

Bu dosya ilk Final omurgasıdır. Bilerek tek notebook içinde tutuldu; ileride stabil hale gelirse `src/` altına modüller ayrılabilir.

## Final Kapsamında Netleştirilmesi Gereken Sorular

Notebook varsayılanlarla çalışacak şekilde hazırlandı. Yine de final rapor/uygulama kalitesi için aşağıdaki kararlar kilitlenmeli:

1. Son kullanıcı hedefi sadece `bir alt zone` mu olacak, yoksa `s1..s5` içinden serbest hedef zone seçimi kesin mi?
2. ACE/FACE aksiyonları fiziksel cihaz komutu gibi mi yorumlanacak, yoksa karar destek önerisi olarak mı kalacak?
3. Bir aksiyonda en fazla kaç feature değişebilir? Varsayılan `MAX_FEATURE_CHANGES = 3`.
4. Bir feature en fazla yüzde kaç azaltılabilir? Varsayılan `MAX_REDUCTION_FRACTION = 0.60`.
5. Konfor maliyeti sentetik ağırlıklarla mı kalacak, yoksa cihaz bazlı gerçek konfor/operasyon maliyeti tanımlanacak mı?
6. Spend-zone esikleri train quantile üzerinden mi kalacak, yoksa iş kuralıyla sabit kWh limitleri mi verilecek?
7. Final model olarak tekil tree model mi istenecek, yoksa AutoGluon ensemble çıktıları tekrar bu notebook'a bağlanacak mı?
8. FACE için tam graph shortest-path zorunlu mu, yoksa veri manifolduna yakın feasible endpoint önerisi final için yeterli mi?
9. DT simülasyonu sadece model-based before/after skor mu gösterecek, yoksa kat/bina bazlı ek animasyon/görselleştirme de istenecek mi?

Bu sorular cevaplanana kadar notebook varsayımlarıyla ilerler.

In [ ]:
from __future__ import annotations

import itertools
import importlib.util
import json
import math
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid, ParameterSampler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.neighbors import kneighbors_graph
from scipy.sparse.csgraph import dijkstra

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:
# -----------------------------
# Global configuration
# -----------------------------

PROJECT_DIR = Path.cwd()
if (PROJECT_DIR / "Final" / "WholeBuilding_EnergyConsumptionDataSet.xlsx").exists():
    PROJECT_DIR = PROJECT_DIR / "Final"
DATA_PATH = PROJECT_DIR / "WholeBuilding_EnergyConsumptionDataSet.xlsx"
SHEET_NAME = "Hourly_Whole Bldg_ByCircuit"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TARGET_SCOPE = "building"  # building, floor1, floor2
K_RECOURSE = 5
MAX_FEATURE_CHANGES = None  # User can change any number of actionable features; search is controlled by cost/beam settings.
MAX_REDUCTION_FRACTION = 0.60
REDUCTION_GRID = np.array([0.10, 0.20, 0.30, 0.40, 0.50, 0.60])
N_SPEND_ZONES = 5
EPS = 1e-6

MODEL_TEMPLATE = GradientBoostingRegressor(random_state=RANDOM_STATE)

BENCHMARK_SCOPES = [TARGET_SCOPE]  # Use ["building", "floor1", "floor2"] for the full longer benchmark.
RUN_HYPERPARAMETER_TUNING = True
TUNING_MAX_CANDIDATES = 1
VALIDATION_RECOURSE_SAMPLE_SIZE = 10
BENCHMARK_MAX_TRAIN_ROWS = 1000
EXPENSIVE_MODEL_MAX_TRAIN_ROWS = 400
AUTOGLUON_TIME_LIMIT_SECONDS = 180
RUN_AUTOGLUON_BENCHMARK = False  # Set True for a longer AutoGluon comparison run.
CATBOOST_OPTUNA_TRIALS = 30
CATBOOST_TUNING_MAX_TRAIN_ROWS = 9000
INCLUDE_HEAVY_MODELS = False  # Set True to include SVR and MLP in the default benchmark.

print(DATA_PATH)


FIXED_ZONE_CONFIG_KWH = {
    "building": [1.0, 1.5, 2.0, 3.0],
    "floor1": [0.2, 0.3, 0.5, 1.2],
    "floor2": [0.35, 0.6, 0.9, 1.6],
}
ZONE_CONFIG_VERSION = "v1_fixed_kwh"
ACE_BEAM_WIDTH = 10
FACE_K_NEIGHBORS = 12
FACE_MAX_ENDPOINTS = 30
DT_SCHEMA_VERSION = "dt_state_v1"


In [ ]:
# -----------------------------
# Feature groups
# -----------------------------

TARGET_COL_RAW = "Hourly Electricity Use (Wh)"
CONTEXT_COLS = ["hour_of_day", "day_of_week", "month", "is_weekend"]

FLOOR1_LOADS = [
    "1st flr AHU",
    "1st flr HP ",
    "1st flr Lights",
    "1st flr Lobby recp",
    "1st flr Office #1 recp",
    "1st flr Office #2 recp",
    "1st flr Office #3 recp",
    "1st flr Bathroom",
    "1st flr Kitchen",
    "1st flr Copy Room recp",
    "1st flr Utility Room recp",
]

FLOOR2_LOADS = [
    "2nd flr AHU - Classroom ",
    "2nd flr AHU - Computer Room",
    "2nd flr HP - Classroom",
    "2nd flr HP - Computer Room",
    "2nd flr Office recp",
    "2nd flr Oven",
    "2nd flr Lights",
    "2nd flr Computer Room recp",
    "2nd flr Classroom #1 recp",
    "2nd flr Classroom #2 recp",
    "2nd flr Bathoom",
    "2nd flr Kitchen",
    "2nd flr Kitchen recp + Dishwasher",
    "2nd flr Water Cooler",
    "2nd flr Computer Room + Kitchen recp",
    "2nd flr Classroom #2 + Copy Room recp",
    "2nd flr Storage Room + Computer Room recp",
]

SHARED_LOADS = ["Refridgerator", "Exterior Lights", "ERV", "Water Heater"]
ALL_LOADS = FLOOR1_LOADS + FLOOR2_LOADS + SHARED_LOADS

SCOPE_LOADS = {
    "building": ALL_LOADS,
    "floor1": FLOOR1_LOADS,
    "floor2": FLOOR2_LOADS,
}

SCOPE_TARGETS = {
    "building": "building_target_t_plus_1_kwh",
    "floor1": "floor1_target_t_plus_1_kwh",
    "floor2": "floor2_target_t_plus_1_kwh",
}


In [ ]:
def intervention_weight(feature_name: str) -> int:
    name = feature_name.lower()
    if "ahu" in name or "hp" in name or "erv" in name:
        return 3
    if "light" in name or "water heater" in name or "oven" in name:
        return 2
    return 1


FEATURE_WEIGHTS = {col: intervention_weight(col) for col in ALL_LOADS}
IMMUTABLE_LOADS = {"Refridgerator"}


def feature_floor(feature_name: str) -> str:
    if feature_name in FLOOR1_LOADS:
        return "floor1"
    if feature_name in FLOOR2_LOADS:
        return "floor2"
    return "shared"


def feature_category(feature_name: str) -> str:
    name = feature_name.lower()
    if "ahu" in name or "hp" in name or "erv" in name:
        return "hvac"
    if "light" in name:
        return "lighting"
    if "refridgerator" in name or "water heater" in name or "water cooler" in name or "oven" in name:
        return "appliance"
    if "recp" in name or "receptacle" in name:
        return "plug_load"
    if "kitchen" in name or "bath" in name:
        return "service_load"
    return "other"


def device_id(feature_name: str) -> str:
    cleaned = feature_name.strip().lower()
    for token in ["#", "+", "-", "(", ")", "/"]:
        cleaned = cleaned.replace(token, " ")
    return "_".join(cleaned.split())


def is_actionable_feature(feature_name: str) -> bool:
    return feature_name not in IMMUTABLE_LOADS


def max_reduction_fraction_for_feature(feature_name: str, hour_of_day: int | None = None) -> float:
    name = feature_name.lower()
    if feature_name in IMMUTABLE_LOADS:
        return 0.0
    if "exterior lights" in name:
        if hour_of_day is not None and (hour_of_day >= 20 or hour_of_day <= 5):
            return 0.25
        return 0.70
    if "ahu" in name or "hp" in name or "erv" in name:
        return 0.30
    if "light" in name:
        return 0.50
    if "water heater" in name or "oven" in name:
        return 0.60
    return 0.80


FEATURE_WEIGHTS


In [ ]:
def reconstruct_timestamp(date_series: pd.Series, time_series: pd.Series) -> pd.Series:
    def to_fractional_day(value: Any) -> float:
        if pd.isna(value):
            return np.nan
        if isinstance(value, (int, float, np.integer, np.floating)):
            return float(value) % 1.0
        if hasattr(value, "hour") and hasattr(value, "minute"):
            seconds = value.hour * 3600 + value.minute * 60 + getattr(value, "second", 0)
            seconds += getattr(value, "microsecond", 0) / 1_000_000
            return seconds / 86400.0
        parsed = pd.to_datetime(value, errors="coerce")
        if not pd.isna(parsed):
            seconds = parsed.hour * 3600 + parsed.minute * 60 + parsed.second + parsed.microsecond / 1_000_000
            return seconds / 86400.0
        parsed_delta = pd.to_timedelta(str(value), errors="coerce")
        if not pd.isna(parsed_delta):
            return parsed_delta / pd.Timedelta(days=1)
        raise ValueError(f"Time value could not be parsed: {value!r}")

    fractional_days = time_series.map(to_fractional_day)
    return (pd.to_datetime(date_series) + pd.to_timedelta(fractional_days, unit="D")).dt.round("s")


def load_and_preprocess(data_path: Path = DATA_PATH) -> pd.DataFrame:
    raw = pd.read_excel(data_path, sheet_name=SHEET_NAME)
    df = raw.dropna().copy()
    df["timestamp"] = reconstruct_timestamp(df["Date"], df["Time"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    for col in ALL_LOADS + [TARGET_COL_RAW]:
        df[col] = pd.to_numeric(df[col], errors="raise") / 1000.0

    df = df.rename(columns={TARGET_COL_RAW: "building_target_t_kwh"})
    df["floor1_target_t_kwh"] = df[FLOOR1_LOADS].sum(axis=1)
    df["floor2_target_t_kwh"] = df[FLOOR2_LOADS].sum(axis=1)
    df["shared_target_t_kwh"] = df[SHARED_LOADS].sum(axis=1)

    df["hour_of_day"] = df["timestamp"].dt.hour.astype(int)
    df["day_of_week"] = df["timestamp"].dt.dayofweek.astype(int)
    df["month"] = df["timestamp"].dt.month.astype(int)
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    for scope, t_col in [("building", "building_target_t_kwh"), ("floor1", "floor1_target_t_kwh"), ("floor2", "floor2_target_t_kwh")]:
        df[SCOPE_TARGETS[scope]] = df[t_col].shift(-1)

    same_row_diff = (df[ALL_LOADS].sum(axis=1) - df["building_target_t_kwh"]).abs().max()
    print(f"Raw rows: {len(raw):,}")
    print(f"Rows after missing drop and t+1 shift: {len(df) - 1:,}")
    print(f"Same-row load sum vs target max diff: {same_row_diff:.3e} kWh")

    return df.iloc[:-1].copy()


base_df = load_and_preprocess()
base_df.head()


In [ ]:
def chronological_split(df: pd.DataFrame, train_ratio: float = 0.70, val_ratio: float = 0.15):
    n = len(df)
    train_end = int(np.floor(n * train_ratio))
    val_end = int(np.floor(n * (train_ratio + val_ratio)))
    return df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(), df.iloc[val_end:].copy()


def scope_columns(scope: str) -> tuple[list[str], list[str], str]:
    load_cols = SCOPE_LOADS[scope]
    feature_cols = load_cols + CONTEXT_COLS
    target_col = SCOPE_TARGETS[scope]
    return load_cols, feature_cols, target_col


train_df, val_df, test_df = chronological_split(base_df)
len(train_df), len(val_df), len(test_df)


In [ ]:
FIXED_ZONE_THRESHOLDS_BY_SCOPE: dict[str, np.ndarray] = {
    scope: np.asarray(thresholds, dtype=float)
    for scope, thresholds in FIXED_ZONE_CONFIG_KWH.items()
}


def build_zone_thresholds(y_train: pd.Series, scope: str, n_zones: int = N_SPEND_ZONES) -> np.ndarray:
    if scope not in FIXED_ZONE_THRESHOLDS_BY_SCOPE:
        raise KeyError(f"No fixed zone threshold configured for scope={scope!r}")
    return FIXED_ZONE_THRESHOLDS_BY_SCOPE[scope]


def assign_zone(values: np.ndarray | pd.Series | float, thresholds: np.ndarray) -> np.ndarray:
    arr = np.asarray(values).reshape(-1)
    return np.searchsorted(thresholds, arr, side="right") + 1


def zone_label(zone_id: int) -> str:
    return f"s{int(zone_id)}"


def lower_target_zone(current_zone: int, requested_zone: int | None = None) -> int:
    if requested_zone is not None:
        return int(max(1, min(N_SPEND_ZONES, requested_zone)))
    return int(max(1, current_zone - 1))


FIXED_ZONE_THRESHOLDS_BY_SCOPE


## Leakage-Safe Model Benchmark ve Final Model Seçimi

Bu bölüm final model seçimini test setine bakmadan yapar.

- Train: model fit ve hiperparametre adayları için kullanılır.
- Validation: model seçimi, zone davranışı ve hızlı recourse/stability sinyali için kullanılır.
- Test: sadece seçilen final modelin son raporlaması için kullanılır.

Karşılaştırılan adaylar: `GBR`, `HistGBR`, `RandomForest`, `ExtraTrees`, `CatBoost`, `LightGBM`, `XGBoost`, `MLP`, `SVR`, `Ridge`, `ElasticNet`. AutoGluon daha uzun süreli benchmark için opsiyonel bir koşu olarak tutulur.

In [ ]:


def evaluate_regression(y_true: pd.Series, y_pred: np.ndarray) -> dict[str, float]:
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(rmse),
        "R2": float(r2_score(y_true, y_pred)),
    }


def optional_import(module_name: str):
    if importlib.util.find_spec(module_name) is None:
        return None
    return __import__(module_name)


xgb = optional_import("xgboost")
lgb = optional_import("lightgbm")
catboost = optional_import("catboost")


def make_model_specs() -> dict[str, tuple[Any, dict[str, list[Any]]]]:
    specs: dict[str, tuple[Any, dict[str, list[Any]]]] = {
        "GBR": (
            GradientBoostingRegressor(random_state=RANDOM_STATE),
            {
                "n_estimators": [50],
                "learning_rate": [0.03, 0.05, 0.10],
                "max_depth": [2, 3, 4],
                "loss": ["squared_error", "absolute_error", "huber"],
            },
        ),
        "HistGBR": (
            HistGradientBoostingRegressor(random_state=RANDOM_STATE),
            {
                "max_iter": [80],
                "learning_rate": [0.03, 0.05, 0.10],
                "max_leaf_nodes": [15, 31, 63],
                "l2_regularization": [0.0, 0.01, 0.1],
            },
        ),
        "RandomForest": (
            RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            {
                "n_estimators": [50],
                "max_depth": [8, 12, None],
                "min_samples_leaf": [1, 3, 5],
            },
        ),
        "ExtraTrees": (
            ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            {
                "n_estimators": [50],
                "max_depth": [8, 12, None],
                "min_samples_leaf": [1, 3, 5],
            },
        ),
        "MLP": (
            Pipeline([
                ("scaler", StandardScaler()),
                ("model", MLPRegressor(random_state=RANDOM_STATE, early_stopping=True, n_iter_no_change=10, max_iter=220)),
            ]),
            {
                "model__hidden_layer_sizes": [(64,)],
                "model__alpha": [0.0001, 0.001, 0.01],
                "model__learning_rate_init": [0.001, 0.003],
            },
        ),
        "SVR": (
            Pipeline([("scaler", StandardScaler()), ("model", SVR())]),
            {
                "model__kernel": ["linear"],
                "model__C": [0.1, 1.0],
                "model__epsilon": [0.05, 0.1],
            },
        ),
        "Ridge": (
            Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
            {"model__alpha": [0.1, 1.0, 10.0, 50.0]},
        ),
        "ElasticNet": (
            Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=20000, random_state=RANDOM_STATE))]),
            {"model__alpha": [0.001, 0.01, 0.1, 1.0], "model__l1_ratio": [0.2, 0.5, 0.8]},
        ),
    }

    if not INCLUDE_HEAVY_MODELS:
        specs.pop("MLP", None)
        specs.pop("SVR", None)

    if catboost is not None:
        specs["CatBoost"] = (
            catboost.CatBoostRegressor(random_seed=RANDOM_STATE, verbose=False, loss_function="RMSE"),
            {
                "iterations": [60],
                "depth": [4, 6, 8],
                "learning_rate": [0.03, 0.05, 0.10],
                "l2_leaf_reg": [1, 3, 5],
            },
        )
    if lgb is not None:
        specs["LightGBM"] = (
            lgb.LGBMRegressor(random_state=RANDOM_STATE, objective="regression", verbosity=-1, n_jobs=-1),
            {
                "n_estimators": [60],
                "learning_rate": [0.03, 0.05, 0.10],
                "num_leaves": [15, 31, 63],
                "min_child_samples": [10, 20, 40],
            },
        )
    if xgb is not None:
        specs["XGBoost"] = (
            xgb.XGBRegressor(random_state=RANDOM_STATE, objective="reg:squarederror", n_jobs=-1, verbosity=0),
            {
                "n_estimators": [60],
                "learning_rate": [0.03, 0.05, 0.10],
                "max_depth": [3, 5, 7],
                "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0],
            },
        )
    return specs


def parameter_candidates(param_grid: dict[str, list[Any]], max_candidates: int = TUNING_MAX_CANDIDATES) -> list[dict[str, Any]]:
    all_candidates = list(ParameterGrid(param_grid)) if param_grid else [{}]
    if not RUN_HYPERPARAMETER_TUNING or len(all_candidates) <= max_candidates:
        return all_candidates
    sampler = ParameterSampler(param_grid, n_iter=max_candidates, random_state=RANDOM_STATE)
    return list(sampler)


def fit_candidate(base_estimator: Any, params: dict[str, Any], X_train: pd.DataFrame, y_train: pd.Series) -> Any:
    estimator = clone(base_estimator)
    estimator.set_params(**params)
    estimator.fit(X_train, y_train)
    return estimator


def quick_intervention_cost(factual_row: pd.Series, candidate_row: pd.Series, changed_features: list[str]) -> float:
    cost = 0.0
    for feature in changed_features:
        reduced = max(0.0, float(factual_row[feature] - candidate_row[feature]))
        denom = max(float(factual_row[feature]), EPS)
        cost += FEATURE_WEIGHTS[feature] * reduced / denom
    return float(cost)


def quick_validation_recourse_metrics(
    estimator: Any,
    scope: str,
    validation_frame: pd.DataFrame,
    feature_cols: list[str],
    load_cols: list[str],
    thresholds: np.ndarray,
    max_samples: int = VALIDATION_RECOURSE_SAMPLE_SIZE,
) -> dict[str, float]:
    preds = estimator.predict(validation_frame[feature_cols])
    zones = assign_zone(preds, thresholds)
    candidate_frame = validation_frame.copy()
    candidate_frame["_pred_zone"] = zones
    candidate_frame = candidate_frame[candidate_frame["_pred_zone"] > 1]
    if candidate_frame.empty:
        return {"recourse_success_rate": 1.0, "recourse_mean_cost": 0.0, "recourse_mean_delta_kwh": 0.0}

    if len(candidate_frame) > max_samples:
        candidate_frame = candidate_frame.sample(max_samples, random_state=RANDOM_STATE).sort_index()

    successes = []
    costs = []
    deltas = []
    actionable_cols = [col for col in load_cols if is_actionable_feature(col)]

    for _, row in candidate_frame.iterrows():
        before_pred = float(estimator.predict(row[feature_cols].astype(float).to_frame().T)[0])
        before_zone = int(assign_zone(before_pred, thresholds)[0])
        desired_zone = max(1, before_zone - 1)
        candidate = row.copy()
        changed = []

        # Greedy operational-realism search: reduce high-current actionable loads first, without a hard feature-count cap.
        ordered_features = sorted(actionable_cols, key=lambda c: float(row[c]), reverse=True)
        for feature in ordered_features:
            cap = max_reduction_fraction_for_feature(feature, int(row["hour_of_day"]))
            if cap <= 0 or float(candidate[feature]) <= EPS:
                continue
            candidate[feature] = max(0.0, float(candidate[feature]) * (1.0 - cap))
            changed.append(feature)
            after_pred = float(estimator.predict(candidate[feature_cols].astype(float).to_frame().T)[0])
            after_zone = int(assign_zone(after_pred, thresholds)[0])
            if after_zone <= desired_zone:
                break

        after_pred = float(estimator.predict(candidate[feature_cols].astype(float).to_frame().T)[0])
        after_zone = int(assign_zone(after_pred, thresholds)[0])
        success = after_zone <= desired_zone
        successes.append(float(success))
        costs.append(quick_intervention_cost(row, candidate, changed) if changed else np.nan)
        deltas.append(before_pred - after_pred)

    valid_costs = [c for c in costs if not np.isnan(c)]
    return {
        "recourse_success_rate": float(np.mean(successes)) if successes else 0.0,
        "recourse_mean_cost": float(np.mean(valid_costs)) if valid_costs else np.inf,
        "recourse_mean_delta_kwh": float(np.mean(deltas)) if deltas else 0.0,
    }


def score_model_on_frame(estimator: Any, frame: pd.DataFrame, feature_cols: list[str], target_col: str, thresholds: np.ndarray) -> dict[str, float]:
    y_true = frame[target_col]
    y_pred = estimator.predict(frame[feature_cols])
    metrics = evaluate_regression(y_true, y_pred)
    pred_zone = assign_zone(y_pred, thresholds)
    true_zone = assign_zone(y_true, thresholds)
    metrics["zone_accuracy"] = float(np.mean(pred_zone == true_zone))
    metrics["mean_ordinal_error"] = float(np.mean(np.abs(pred_zone - true_zone)))
    return metrics


def benchmark_models_for_scope(scope: str) -> tuple[pd.DataFrame, dict[str, Any]]:
    load_cols, feature_cols, target_col = scope_columns(scope)
    thresholds = build_zone_thresholds(train_df[target_col], scope)
    specs = make_model_specs()
    records: list[dict[str, Any]] = []
    fitted: dict[str, Any] = {}

    benchmark_train_df = train_df.iloc[-BENCHMARK_MAX_TRAIN_ROWS:].copy() if len(train_df) > BENCHMARK_MAX_TRAIN_ROWS else train_df
    X_val = val_df[feature_cols]
    y_val = val_df[target_col]

    for model_name, (base_estimator, param_grid) in specs.items():
        model_train_df = benchmark_train_df
        if model_name in {"SVR", "MLP"} and len(model_train_df) > EXPENSIVE_MODEL_MAX_TRAIN_ROWS:
            model_train_df = model_train_df.iloc[-EXPENSIVE_MODEL_MAX_TRAIN_ROWS:].copy()
        X_train = model_train_df[feature_cols]
        y_train = model_train_df[target_col]
        best_estimator = None
        best_params = None
        best_val_mae = np.inf
        candidates = parameter_candidates(param_grid)

        for params in candidates:
            try:
                estimator = fit_candidate(base_estimator, params, X_train, y_train)
                val_pred = estimator.predict(X_val)
                val_mae = mean_absolute_error(y_val, val_pred)
            except Exception as exc:
                print(f"{scope} / {model_name} skipped params {params}: {exc}")
                continue
            if val_mae < best_val_mae:
                best_val_mae = val_mae
                best_estimator = estimator
                best_params = params

        if best_estimator is None:
            continue

        val_metrics = score_model_on_frame(best_estimator, val_df, feature_cols, target_col, thresholds)
        recourse_metrics = quick_validation_recourse_metrics(best_estimator, scope, val_df, feature_cols, load_cols, thresholds)
        record = {
            "scope": scope,
            "model_name": model_name,
            "best_params": best_params,
            "zone_thresholds_kwh": thresholds.tolist(),
            "candidate_count": len(candidates),
            **{f"val_{k}": v for k, v in val_metrics.items()},
            **recourse_metrics,
        }
        records.append(record)
        fitted[model_name] = best_estimator

    result = pd.DataFrame(records)
    if result.empty:
        raise RuntimeError(f"No model could be trained for scope={scope}")

    result["prediction_rank"] = (
        result["val_MAE"].rank(ascending=True)
        + result["val_RMSE"].rank(ascending=True)
        + result["val_R2"].rank(ascending=False)
    ) / 3
    result["zone_rank"] = (
        result["val_zone_accuracy"].rank(ascending=False)
        + result["val_mean_ordinal_error"].rank(ascending=True)
    ) / 2
    result["recourse_rank"] = result["recourse_success_rate"].rank(ascending=False)
    result["stability_rank"] = result["recourse_mean_cost"].replace(np.inf, 1e9).rank(ascending=True)
    result["selection_rank"] = result[["prediction_rank", "zone_rank", "recourse_rank", "stability_rank"]].mean(axis=1)
    result = result.sort_values(["selection_rank", "val_MAE", "val_mean_ordinal_error"]).reset_index(drop=True)

    selected_model_name = result.iloc[0]["model_name"]
    return result, {"model_name": selected_model_name, "estimator": fitted[selected_model_name], "thresholds": thresholds}


benchmark_tables: dict[str, pd.DataFrame] = {}
selected_validation_models: dict[str, dict[str, Any]] = {}
for scope in BENCHMARK_SCOPES:
    print(f"\n=== Benchmark: {scope} ===")
    table, selected = benchmark_models_for_scope(scope)
    benchmark_tables[scope] = table
    selected_validation_models[scope] = selected
    display(table[[
        "scope", "model_name", "selection_rank", "val_MAE", "val_RMSE", "val_R2",
        "val_zone_accuracy", "val_mean_ordinal_error", "recourse_success_rate", "recourse_mean_cost",
    ]].head(12))

benchmark_df = pd.concat(benchmark_tables.values(), ignore_index=True)
benchmark_df.to_csv(OUTPUT_DIR / "model_benchmark_validation.csv", index=False)
selected_validation_summary = pd.DataFrame([
    {"scope": scope, "selected_model": selected["model_name"], "zone_thresholds_kwh": selected["thresholds"].tolist()}
    for scope, selected in selected_validation_models.items()
])
selected_validation_summary.to_csv(OUTPUT_DIR / "selected_model_validation_summary.csv", index=False)
selected_validation_summary


### Opsiyonel AutoGluon Benchmark

AutoGluon koşusu daha uzun sürebilir. `RUN_AUTOGLUON_BENCHMARK = True` yapılırsa aynı kronolojik train/validation ayrımı ile AutoGluon da benchmark'a eklenebilir. Final kararında AutoGluon ensemble benchmark olarak tutulur; recourse için mümkünse en iyi tekil tree model tercih edilir.

In [ ]:
if RUN_AUTOGLUON_BENCHMARK:
    from autogluon.tabular import TabularPredictor

    autogluon_records = []
    for scope in BENCHMARK_SCOPES:
        load_cols, feature_cols, target_col = scope_columns(scope)
        ag_train = train_df[feature_cols + [target_col]].copy()
        ag_val = val_df[feature_cols + [target_col]].copy()
        save_path = OUTPUT_DIR / "autogluon" / scope
        predictor = TabularPredictor(
            label=target_col,
            problem_type="regression",
            eval_metric="mean_absolute_error",
            path=str(save_path),
            verbosity=0,
        )
        predictor.fit(
            train_data=ag_train,
            tuning_data=ag_val,
            presets="medium_quality",
            time_limit=AUTOGLUON_TIME_LIMIT_SECONDS,
        )
        leaderboard = predictor.leaderboard(ag_val, silent=True)
        leaderboard.to_csv(OUTPUT_DIR / f"autogluon_{scope}_leaderboard.csv", index=False)
        autogluon_records.append({"scope": scope, "best_model": predictor.model_best, "path": str(save_path)})
    pd.DataFrame(autogluon_records)
else:
    print("AutoGluon benchmark skipped. Set RUN_AUTOGLUON_BENCHMARK=True to run it.")


## CatBoost-only Optuna Hiperparametre Optimizasyonu

Bu bölüm benchmark sonucunda güçlü aday olan CatBoost'u daha ciddi bir arama ile optimize eder.

Leakage guardrail:

- Optuna trial fit işlemi yalnızca `train_df` üzerinde yapılır.
- Objective yalnızca `val_df` üzerinde hesaplanır.
- `test_df` Optuna tarafında hiç kullanılmaz.
- En iyi parametreler bulunduktan sonra final CatBoost modeli `train+validation` üzerinde tekrar fit edilir ve test set sadece son raporlama için kullanılır.

Objective ana sinyali validation MAE'dir. Zone ve recourse metrikleri seçim tablosunda saklanır; final raporda birlikte yorumlanır.

In [ ]:
def tune_catboost_with_optuna(scope: str, n_trials: int = CATBOOST_OPTUNA_TRIALS) -> tuple[pd.DataFrame, dict[str, Any]]:
    if catboost is None:
        raise RuntimeError("catboost is not installed")

    load_cols, feature_cols, target_col = scope_columns(scope)
    thresholds = build_zone_thresholds(train_df[target_col], scope)
    tune_train_df = train_df.iloc[-CATBOOST_TUNING_MAX_TRAIN_ROWS:].copy() if len(train_df) > CATBOOST_TUNING_MAX_TRAIN_ROWS else train_df
    X_train = tune_train_df[feature_cols]
    y_train = tune_train_df[target_col]
    X_val = val_df[feature_cols]
    y_val = val_df[target_col]

    trial_records: list[dict[str, Any]] = []

    def objective(trial: optuna.Trial) -> float:
        params = {
            "loss_function": trial.suggest_categorical("loss_function", ["RMSE", "MAE"]),
            "eval_metric": "MAE",
            "iterations": trial.suggest_int("iterations", 200, 900, step=100),
            "depth": trial.suggest_int("depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 12.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 0.1, 3.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.5),
            "border_count": trial.suggest_categorical("border_count", [32, 64, 128]),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 30),
            "random_seed": RANDOM_STATE,
            "verbose": False,
            "allow_writing_files": False,
        }
        model = catboost.CatBoostRegressor(**params)
        model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True, early_stopping_rounds=80, verbose=False)
        val_pred = model.predict(X_val)
        metrics = evaluate_regression(y_val, val_pred)
        pred_zone = assign_zone(val_pred, thresholds)
        true_zone = assign_zone(y_val, thresholds)
        metrics["zone_accuracy"] = float(np.mean(pred_zone == true_zone))
        metrics["mean_ordinal_error"] = float(np.mean(np.abs(pred_zone - true_zone)))
        recourse_metrics = quick_validation_recourse_metrics(model, scope, val_df, feature_cols, load_cols, thresholds)

        trial_record = {
            "trial": trial.number,
            "scope": scope,
            "params": params,
            **{f"val_{k}": v for k, v in metrics.items()},
            **recourse_metrics,
        }
        trial_records.append(trial_record)

        trial.set_user_attr("val_RMSE", metrics["RMSE"])
        trial.set_user_attr("val_R2", metrics["R2"])
        trial.set_user_attr("zone_accuracy", metrics["zone_accuracy"])
        trial.set_user_attr("mean_ordinal_error", metrics["mean_ordinal_error"])
        trial.set_user_attr("recourse_success_rate", recourse_metrics["recourse_success_rate"])
        return metrics["MAE"]

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="minimize", sampler=sampler, study_name=f"catboost_{scope}_h_t_plus_1")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    trials_df = pd.DataFrame(trial_records).sort_values("val_MAE").reset_index(drop=True)
    best_params = study.best_trial.params.copy()
    best_params.update({
        "eval_metric": "MAE",
        "random_seed": RANDOM_STATE,
        "verbose": False,
        "allow_writing_files": False,
    })

    best_model = catboost.CatBoostRegressor(**best_params)
    best_model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True, early_stopping_rounds=80, verbose=False)
    val_metrics = score_model_on_frame(best_model, val_df, feature_cols, target_col, thresholds)
    recourse_metrics = quick_validation_recourse_metrics(best_model, scope, val_df, feature_cols, load_cols, thresholds)

    trials_df.to_csv(OUTPUT_DIR / f"{scope}_catboost_optuna_trials.csv", index=False)
    best_payload = {
        "scope": scope,
        "n_trials": n_trials,
        "best_value_val_mae": float(study.best_value),
        "best_params": best_params,
        "validation_metrics": val_metrics,
        "validation_recourse_metrics": recourse_metrics,
        "zone_thresholds_kwh": thresholds.tolist(),
    }
    (OUTPUT_DIR / f"{scope}_catboost_optuna_best.json").write_text(json.dumps(best_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    return trials_df, {"model_name": "CatBoost_Optuna", "estimator": best_model, "thresholds": thresholds, "best_payload": best_payload}


catboost_optuna_trials: dict[str, pd.DataFrame] = {}
catboost_optuna_selected: dict[str, dict[str, Any]] = {}
for scope in BENCHMARK_SCOPES:
    print(f"\n=== CatBoost Optuna tuning: {scope} ({CATBOOST_OPTUNA_TRIALS} trials) ===")
    trials_df, selected = tune_catboost_with_optuna(scope, CATBOOST_OPTUNA_TRIALS)
    catboost_optuna_trials[scope] = trials_df
    catboost_optuna_selected[scope] = selected
    selected_validation_models[scope] = selected
    display(trials_df[[
        "trial", "val_MAE", "val_RMSE", "val_R2", "val_zone_accuracy",
        "val_mean_ordinal_error", "recourse_success_rate", "recourse_mean_cost",
    ]].head(10))

pd.DataFrame([
    {
        "scope": scope,
        "selected_model": selected["model_name"],
        "best_value_val_mae": selected["best_payload"]["best_value_val_mae"],
        "zone_thresholds_kwh": selected["thresholds"].tolist(),
    }
    for scope, selected in catboost_optuna_selected.items()
]).to_csv(OUTPUT_DIR / "catboost_optuna_selection_summary.csv", index=False)


In [ ]:
@dataclass
class ScopeModel:
    scope: str
    model: Any
    load_cols: list[str]
    feature_cols: list[str]
    target_col: str
    thresholds: np.ndarray
    metrics: dict[str, float]
    selected_model_name: str


def train_scope_model(scope: str) -> ScopeModel:
    load_cols, feature_cols, target_col = scope_columns(scope)
    selected = selected_validation_models[scope]
    selected_name = selected["model_name"]
    validation_estimator = selected["estimator"]

    # Final model is refit on train+validation only. Test remains untouched until final scoring.
    model = clone(validation_estimator)
    train_val_frame = pd.concat([train_df, val_df], axis=0)
    model.fit(train_val_frame[feature_cols], train_val_frame[target_col])

    thresholds = build_zone_thresholds(train_df[target_col], scope)
    y_pred = model.predict(test_df[feature_cols])
    metrics = evaluate_regression(test_df[target_col], y_pred)

    pred_zone = assign_zone(y_pred, thresholds)
    true_zone = assign_zone(test_df[target_col], thresholds)
    metrics["zone_accuracy"] = float(np.mean(pred_zone == true_zone))
    metrics["mean_ordinal_error"] = float(np.mean(np.abs(pred_zone - true_zone)))

    return ScopeModel(scope, model, load_cols, feature_cols, target_col, thresholds, metrics, selected_name)


scope_models = {scope: train_scope_model(scope) for scope in BENCHMARK_SCOPES}
final_model_test_metrics = pd.DataFrame([
    {"scope": s, "selected_model": m.selected_model_name, "zone_thresholds_kwh": m.thresholds.tolist(), **m.metrics}
    for s, m in scope_models.items()
])
final_model_test_metrics.to_csv(OUTPUT_DIR / "final_model_test_metrics.csv", index=False)
final_model_test_metrics


In [ ]:
def prediction_record(scope_model: ScopeModel, row: pd.Series) -> dict[str, Any]:
    X = row[scope_model.feature_cols].astype(float).to_frame().T
    pred = float(scope_model.model.predict(X)[0])
    zone = int(assign_zone(pred, scope_model.thresholds)[0])
    return {"predicted_kwh": pred, "predicted_zone": zone}


scope_model = scope_models[TARGET_SCOPE]
factual_idx = test_df.index[len(test_df) // 2]
factual = test_df.loc[factual_idx].copy()
factual_prediction = prediction_record(scope_model, factual)
target_zone = lower_target_zone(factual_prediction["predicted_zone"])

factual_prediction, target_zone, factual[["timestamp"] + scope_model.feature_cols].to_frame().T


In [ ]:
def intervention_cost(factual_row: pd.Series, candidate_row: pd.Series, changed_features: list[str]) -> float:
    cost = 0.0
    for feature in changed_features:
        reduced = max(0.0, float(factual_row[feature] - candidate_row[feature]))
        denom = max(float(factual_row[feature]), EPS)
        cost += FEATURE_WEIGHTS[feature] * reduced / denom
    return float(cost)


def candidate_summary(method: str, scope_model: ScopeModel, factual_row: pd.Series, candidate_row: pd.Series, changed_features: list[str]) -> dict[str, Any]:
    before = prediction_record(scope_model, factual_row)
    after = prediction_record(scope_model, candidate_row)
    reductions = {f: float(factual_row[f] - candidate_row[f]) for f in changed_features}
    return {
        "method": method,
        "scope": scope_model.scope,
        "before_kwh": before["predicted_kwh"],
        "after_kwh": after["predicted_kwh"],
        "before_zone": before["predicted_zone"],
        "after_zone": after["predicted_zone"],
        "delta_kwh": before["predicted_kwh"] - after["predicted_kwh"],
        "changed_count": len(changed_features),
        "cost": intervention_cost(factual_row, candidate_row, changed_features),
        "changed_features": changed_features,
        "reductions_kwh": reductions,
    }


def is_successful(summary: dict[str, Any], desired_zone: int) -> bool:
    return int(summary["after_zone"]) <= int(desired_zone)


## ACE-benzeri Recourse

Bu implementasyon actionable feature'larda sadece azaltım yapar. Adaylar `MAX_FEATURE_CHANGES` sınırına göre üretilir, modelle tekrar skorlanır ve hedef zone'a inenler maliyet/seyreklik/enerji düşüşü ile sıralanır.

In [ ]:
def ace_like_recourse(
    scope_model: ScopeModel,
    factual_row: pd.Series,
    desired_zone: int,
    top_k: int = K_RECOURSE,
    beam_width: int = ACE_BEAM_WIDTH,
) -> pd.DataFrame:
    actionable = [col for col in scope_model.load_cols if is_actionable_feature(col) and factual_row[col] > EPS]
    actionable = sorted(actionable, key=lambda c: factual_row[c], reverse=True)[:12]
    if not actionable:
        return pd.DataFrame()

    def score_summary(summary: dict[str, Any]) -> tuple:
        return (
            0 if summary["success"] else 1,
            summary["after_zone"],
            summary["cost"],
            summary["changed_count"],
            -summary["delta_kwh"],
        )

    initial = factual_row.copy()
    beam: list[tuple[pd.Series, list[str]]] = [(initial, [])]
    collected: list[dict[str, Any]] = []

    for _step in range(len(actionable)):
        expanded: list[tuple[tuple, pd.Series, list[str], dict[str, Any]]] = []
        for state, changed in beam:
            remaining = [feature for feature in actionable if feature not in changed]
            for feature in remaining:
                cap = max_reduction_fraction_for_feature(feature, int(factual_row["hour_of_day"]))
                if cap <= 0:
                    continue
                candidate = state.copy()
                candidate[feature] = max(0.0, float(candidate[feature]) * (1.0 - cap))
                candidate_changed = changed + [feature]
                summary = candidate_summary("ACE_BEAM", scope_model, factual_row, candidate, candidate_changed)
                summary["success"] = is_successful(summary, desired_zone)
                expanded.append((score_summary(summary), candidate, candidate_changed, summary))

        if not expanded:
            break
        expanded = sorted(expanded, key=lambda item: item[0])[:beam_width]
        beam = [(candidate, changed) for _score, candidate, changed, _summary in expanded]
        collected.extend(summary for _score, _candidate, _changed, summary in expanded)
        if sum(1 for _score, _candidate, _changed, summary in expanded if summary["success"]) >= top_k:
            break

    result = pd.DataFrame(collected)
    if result.empty:
        return result
    result = result.drop_duplicates(subset=["after_kwh", "cost", "changed_count"])
    result = result.sort_values(["success", "after_zone", "cost", "changed_count", "after_kwh"], ascending=[False, True, True, True, True])
    return result.head(top_k).reset_index(drop=True)


ace_results = ace_like_recourse(scope_model, factual, target_zone)
ace_results[["method", "before_kwh", "after_kwh", "before_zone", "after_zone", "delta_kwh", "changed_count", "cost", "changed_features", "success"]]


## FACE-benzeri Recourse

FACE mantığı için train verisi üzerinde manifold temsili kullanılır. Bu ilk sürümde hedef zone'a düşen, factual örneğe göre sadece azaltım gerektiren ve scaled actionable uzayda yakın olan feasible endpoint'ler önerilir.

Not: Tam FACE shortest-path graph versiyonu istenirse bu fonksiyon `kneighbors_graph + dijkstra` ile genişletilecek.

In [ ]:
def face_like_recourse(
    scope_model: ScopeModel,
    factual_row: pd.Series,
    desired_zone: int,
    top_k: int = K_RECOURSE,
) -> pd.DataFrame:
    def face_failure_frame(reason: str) -> pd.DataFrame:
        summary = candidate_summary("FACE_GRAPH", scope_model, factual_row, factual_row.copy(), [])
        summary["success"] = False
        summary["graph_distance"] = np.nan
        summary["endpoint_index"] = None
        summary["failure_reason"] = reason
        return pd.DataFrame([summary])

    load_cols = [col for col in scope_model.load_cols if is_actionable_feature(col)]
    if not load_cols:
        return face_failure_frame("no_actionable_load_features")

    scaler = StandardScaler().fit(train_df[load_cols])
    train_work = train_df.copy().reset_index(drop=True)
    train_pred = scope_model.model.predict(train_work[scope_model.feature_cols])
    train_work["pred_kwh"] = train_pred
    train_work["pred_zone"] = assign_zone(train_pred, scope_model.thresholds)

    reduction_only_mask = (train_work[load_cols] <= factual_row[load_cols].to_numpy() + EPS).all(axis=1)
    target_mask = train_work["pred_zone"] <= desired_zone
    endpoints = train_work[reduction_only_mask & target_mask].copy()
    if endpoints.empty:
        return face_failure_frame("no_reduction_only_train_endpoint_in_target_zone")

    X_scaled = scaler.transform(train_work[load_cols])
    factual_scaled = scaler.transform(factual_row[load_cols].astype(float).to_frame().T)
    nearest = NearestNeighbors(n_neighbors=1).fit(X_scaled)
    start_node = int(nearest.kneighbors(factual_scaled, return_distance=False)[0, 0])

    graph = kneighbors_graph(
        X_scaled,
        n_neighbors=min(FACE_K_NEIGHBORS, len(train_work) - 1),
        mode="distance",
        include_self=False,
    )
    graph = graph.maximum(graph.T)
    distances, predecessors = dijkstra(graph, directed=False, indices=start_node, return_predecessors=True)

    endpoints["graph_distance"] = distances[endpoints.index.to_numpy()]
    endpoints = endpoints[np.isfinite(endpoints["graph_distance"])].sort_values("graph_distance").head(FACE_MAX_ENDPOINTS)
    if endpoints.empty:
        return face_failure_frame("no_reachable_graph_endpoint")

    summaries: list[dict[str, Any]] = []
    for endpoint_idx, endpoint in endpoints.iterrows():
        candidate = factual_row.copy()
        changed_features = []
        for feature in load_cols:
            new_value = min(float(factual_row[feature]), float(endpoint[feature]))
            if factual_row[feature] - new_value > EPS:
                candidate[feature] = new_value
                changed_features.append(feature)

        summary = candidate_summary("FACE_GRAPH", scope_model, factual_row, candidate, changed_features)
        summary["success"] = is_successful(summary, desired_zone)
        summary["graph_distance"] = float(endpoint["graph_distance"])
        summary["endpoint_index"] = int(endpoint_idx)
        summaries.append(summary)

    result = pd.DataFrame(summaries)
    if result.empty:
        return result
    result = result.sort_values(["success", "after_zone", "graph_distance", "cost"], ascending=[False, True, True, True])
    return result.head(top_k).reset_index(drop=True)


face_results = face_like_recourse(scope_model, factual, target_zone)
face_display_cols = ["method", "before_kwh", "after_kwh", "before_zone", "after_zone", "delta_kwh", "changed_count", "cost", "graph_distance", "changed_features", "success"]
if face_results.empty:
    print("FACE: no result frame returned.")
else:
    display_cols = [c for c in face_display_cols + ["failure_reason"] if c in face_results.columns]
    display(face_results[display_cols])


## Dijital Twin Simülasyonu

DT burada fiziksel bina yerine model tabanlı sanal ikiz olarak ele alınır. Factual state ve recourse uygulanmış counterfactual state aynı predictor ile tekrar skorlanır; before/after enerji ve zone farkı raporlanır.

In [ ]:
def combine_recourse_outputs(*frames: pd.DataFrame) -> pd.DataFrame:
    available = [frame for frame in frames if frame is not None and not frame.empty]
    if not available:
        return pd.DataFrame()
    cols = ["method", "before_kwh", "after_kwh", "before_zone", "after_zone", "delta_kwh", "changed_count", "cost", "graph_distance", "endpoint_index", "failure_reason", "changed_features", "reductions_kwh", "success"]
    combined = pd.concat(available, ignore_index=True, sort=False)
    combined = combined.sort_values(["success", "after_zone", "cost", "delta_kwh"], ascending=[False, True, True, False])
    return combined[[c for c in cols if c in combined.columns]].reset_index(drop=True)


dt_results = combine_recourse_outputs(ace_results, face_results)
dt_results.to_csv(OUTPUT_DIR / f"{TARGET_SCOPE}_dt_recourse_results.csv", index=False)
dt_results.head(10)


In [ ]:
if not dt_results.empty:
    plot_df = dt_results.head(K_RECOURSE).copy()
    labels = [f"{m}-{i+1}" for i, m in enumerate(plot_df["method"])]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(labels, plot_df["before_kwh"], label="Before", alpha=0.65)
    axes[0].bar(labels, plot_df["after_kwh"], label="After", alpha=0.85)
    axes[0].set_title(f"DT before/after predicted energy - {TARGET_SCOPE}")
    axes[0].set_ylabel("kWh")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].legend()

    axes[1].bar(labels, plot_df["cost"], color="#8f5f2f")
    axes[1].set_title("Synthetic intervention cost")
    axes[1].set_ylabel("cost")
    axes[1].tick_params(axis="x", rotation=30)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{TARGET_SCOPE}_dt_recourse_summary.png", dpi=160)
    plt.show()
else:
    print("No recourse candidates found for the selected factual row and target zone.")


In [ ]:
def device_scene_position(feature_name: str, index_by_floor: dict[str, int]) -> dict[str, float]:
    floor = feature_floor(feature_name)
    idx = index_by_floor.get(floor, 0)
    index_by_floor[floor] = idx + 1
    y = 0.0 if floor in {"floor1", "shared"} else 1.25
    return {
        "x": float((idx % 6) * 1.4 - 3.5),
        "y": float(y),
        "z": float((idx // 6) * 1.2 - 1.8),
    }


def build_device_catalog(row: pd.Series) -> list[dict[str, Any]]:
    index_by_floor = {"floor1": 0, "floor2": 0, "shared": 0}
    devices = []
    for feature in ALL_LOADS:
        hour = int(row["hour_of_day"])
        devices.append({
            "id": device_id(feature),
            "label": feature.strip(),
            "source_column": feature,
            "floor": feature_floor(feature),
            "category": feature_category(feature),
            "current_kwh": float(row[feature]),
            "actionable": bool(is_actionable_feature(feature)),
            "immutable": bool(not is_actionable_feature(feature)),
            "max_auto_reduction_fraction": float(max_reduction_fraction_for_feature(feature, hour)),
            "comfort_weight": int(FEATURE_WEIGHTS[feature]),
            "scene": device_scene_position(feature, index_by_floor),
        })
    return devices


def selected_recourse_row(dt_frame: pd.DataFrame) -> dict[str, Any] | None:
    if dt_frame.empty:
        return None
    successful = dt_frame[dt_frame["success"] == True]
    selected = successful.iloc[0] if not successful.empty else dt_frame.iloc[0]
    return selected.to_dict()


def parse_reductions(value: Any) -> dict[str, float]:
    if isinstance(value, dict):
        return {str(k): float(v) for k, v in value.items()}
    if isinstance(value, str) and value.strip():
        try:
            import ast
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                return {str(k): float(v) for k, v in parsed.items()}
        except Exception:
            return {}
    return {}


def build_recourse_actions(selected: dict[str, Any] | None) -> list[dict[str, Any]]:
    if not selected:
        return []
    reductions = parse_reductions(selected.get("reductions_kwh", {}))
    actions = []
    for feature, delta in reductions.items():
        before = float(factual[feature])
        after = max(0.0, before - float(delta))
        action_type = "turn_off" if after <= EPS else "reduce_percent"
        actions.append({
            "feature": feature,
            "device_id": device_id(feature),
            "action_type": action_type,
            "reduction_fraction": float(delta / max(before, EPS)),
            "before_kwh": before,
            "after_kwh": after,
            "delta_kwh": float(delta),
            "cost_weight": int(FEATURE_WEIGHTS[feature]),
        })
    return actions


selected_recourse = selected_recourse_row(dt_results)
recourse_actions = build_recourse_actions(selected_recourse)

floor_states = []
for floor_id, cols in [("floor1", FLOOR1_LOADS), ("floor2", FLOOR2_LOADS)]:
    floor_current = float(factual[cols].sum())
    floor_states.append({
        "id": floor_id,
        "label": "1st Floor" if floor_id == "floor1" else "2nd Floor",
        "current_kwh": floor_current,
        "zone": zone_label(int(assign_zone(floor_current, build_zone_thresholds(train_df[SCOPE_TARGETS[floor_id]], floor_id))[0])),
        "device_ids": [device_id(col) for col in cols],
    })

shared_current = float(factual[SHARED_LOADS].sum())

dt_state = {
    "schema_version": DT_SCHEMA_VERSION,
    "timestamp": pd.Timestamp(factual["timestamp"]).isoformat(),
    "scope": TARGET_SCOPE,
    "unit": "kWh",
    "zone_config_version": ZONE_CONFIG_VERSION,
    "zone_config": FIXED_ZONE_CONFIG_KWH,
    "model": {
        "name": scope_model.selected_model_name,
        "target": scope_model.target_col,
        "prediction_horizon": "t+1",
        "metrics": scope_model.metrics,
    },
    "building": {
        "current_total_kwh": float(factual[ALL_LOADS].sum()),
        "predicted_t_plus_1_kwh": float(factual_prediction["predicted_kwh"]),
        "predicted_zone": zone_label(int(factual_prediction["predicted_zone"])),
        "target_zone": zone_label(int(target_zone)),
    },
    "floors": floor_states,
    "shared_loads": {
        "current_kwh": shared_current,
        "device_ids": [device_id(col) for col in SHARED_LOADS],
    },
    "devices": build_device_catalog(factual),
    "recourse": {
        "method": selected_recourse.get("method") if selected_recourse else None,
        "success": bool(selected_recourse.get("success")) if selected_recourse else False,
        "before": {
            "predicted_kwh": float(selected_recourse.get("before_kwh")) if selected_recourse else float(factual_prediction["predicted_kwh"]),
            "zone": zone_label(int(selected_recourse.get("before_zone"))) if selected_recourse else zone_label(int(factual_prediction["predicted_zone"])),
        },
        "after": {
            "predicted_kwh": float(selected_recourse.get("after_kwh")) if selected_recourse else None,
            "zone": zone_label(int(selected_recourse.get("after_zone"))) if selected_recourse else None,
        },
        "actions": recourse_actions,
        "total_cost": float(selected_recourse.get("cost")) if selected_recourse else None,
        "changed_feature_count": int(selected_recourse.get("changed_count")) if selected_recourse else 0,
    },
    "manual_override": {
        "enabled": True,
        "supported_actions": ["reduce_percent", "turn_off", "restore"],
    },
}

(OUTPUT_DIR / "dt_state_v1.json").write_text(json.dumps(dt_state, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(dt_state["devices"]).to_csv(OUTPUT_DIR / "dt_device_catalog.csv", index=False)

summary = {
    "target_scope": TARGET_SCOPE,
    "factual_index": int(factual_idx),
    "factual_timestamp": str(factual["timestamp"]),
    "factual_prediction": factual_prediction,
    "target_zone": int(target_zone),
    "selected_models": {scope: model.selected_model_name for scope, model in scope_models.items()},
    "model_metrics": {scope: model.metrics for scope, model in scope_models.items()},
    "zone_thresholds_kwh": {scope: model.thresholds.tolist() for scope, model in scope_models.items()},
    "dt_state_json": str(OUTPUT_DIR / "dt_state_v1.json"),
    "device_catalog_csv": str(OUTPUT_DIR / "dt_device_catalog.csv"),
    "output_csv": str(OUTPUT_DIR / f"{TARGET_SCOPE}_dt_recourse_results.csv"),
    "output_plot": str(OUTPUT_DIR / f"{TARGET_SCOPE}_dt_recourse_summary.png"),
}

(OUTPUT_DIR / f"{TARGET_SCOPE}_final_run_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
dt_state


## Sonraki Uygulama Adımları

- Kullanıcı hedef zone seçimi için bir parametre/form alanı eklenecek.
- ACE beam search ve FACE kNN shortest-path sonuçları DT JSON şemasına bağlandı; FACE infeasible ise diagnostik satır olarak kaydedilir.
- ACE aday üretimi daha akıllı hale getirilecek: feature importance, SHAP etkisi veya gradient-free optimizer ile arama daraltılabilir.
- DT görselleştirme bina/kat ayrımı, zone timeline, action waterfall ve 3D scene temsili ile genişletilecek.
- Raporlama için notebook çıktıları PDF/HTML olarak export edilecek.